In [1]:
%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import _pickle as cPickle
import nltk

## Read prepared data

In [2]:
df_start = pd.read_csv('df_start.tsv', sep=',')

In [3]:
df_start

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
0,1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
1,2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
2,3,0,2,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
3,4,0,2,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
4,5,0,2,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
...,...,...,...,...,...,...,...,...,...
599630,599631,291,76245,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13
599631,599632,291,76246,2018-04-06 13:20:12,Settings,Opened,0 days 00:26:59,2018-04-06 13:20:12,2018-04-06 13:20:14
599632,599633,291,76247,2018-04-06 14:34:15,Settings,Opened,0 days 01:14:01,2018-04-06 14:34:15,2018-04-06 14:34:17
599633,599634,291,76247,2018-04-06 14:34:34,Facebook,Opened,0 days 00:00:17,2018-04-06 14:34:34,2018-04-06 14:35:37


## App usage records

In [4]:
len(df_start)

599635

## Sessions

In [5]:
len(df_start['session_id'].unique())

67309

## Unique apps

In [6]:
len(df_start['app_name'].unique())

87

## Users

In [7]:
len(df_start['user_id'].unique())

292

## Mean duration/user 

In [8]:
df_start.loc[:,'timestamp'] = pd.to_datetime(df_start['timestamp'])

grouped = df_start.groupby('user_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each user
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the mean duration
mean_duration_seconds = grouped['duration_seconds'].mean()

# Convert the mean duration back to a Timedelta object for better readability
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')

print(mean_duration)

15 days 16:29:39.106164383


## Mean session time length

In [9]:
grouped = df_start.groupby('session_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each session
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the mean duration
mean_duration_seconds = grouped['duration_seconds'].mean()

# Compute the standard deviation
std_duration_seconds = grouped['duration_seconds'].std()

# Convert the mean duration back to a Timedelta object for better readability
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')
std_duration = pd.to_timedelta(std_duration_seconds, unit='s')

print("Mean session time length:", mean_duration)
print("Standard Deviation of session time length:", std_duration)

Mean session time length: 0 days 00:04:42.882348572
Standard Deviation of session time length: 0 days 00:08:57.048502996


## Median session time length

In [10]:

grouped = df_start.groupby('session_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each session
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the median duration
median_duration_seconds = grouped['duration_seconds'].median()

median_duration = pd.to_timedelta(median_duration_seconds, unit='s')

print("Median Duration:", median_duration)

Median Duration: 0 days 00:00:56


## Mean unique apps in each session

In [11]:
unique_apps_per_session = df_start.groupby('session_id')['app_name'].nunique()

mean_unique_apps = unique_apps_per_session.mean()
std_unique_apps = unique_apps_per_session.std()

print("Mean number of unique apps per session:", mean_unique_apps)
print("Standard Deviation of unique apps per session:", std_unique_apps)

Mean number of unique apps per session: 2.503498789166382
Standard Deviation of unique apps per session: 1.6953192845603906


## Median unique apps in each session

In [12]:
median_unique_apps = unique_apps_per_session.median()
print("Median number of unique apps per session:", median_unique_apps)

Median number of unique apps per session: 2.0


## Mean app switches within a session

In [13]:
df_start['app_switch'] = (df_start['app_name'] != df_start['app_name'].shift()).astype(int)

session_switches = df_start.groupby('session_id')['app_switch'].sum()

mean_switches = session_switches.mean()
std_switches = session_switches.std()

print("Average (mean) of session switches:", mean_switches)
print("Standard deviation of session switches:", std_switches)

Average (mean) of session switches: 8.335616336596889
Standard deviation of session switches: 23.361845241386447


## Median app switches within a session

In [14]:
median_switches = session_switches.median()

print("Median of session switches:", median_switches)

Median of session switches: 2.0
